# 📘 Data Governance & Unity Catalog
## Databricks Notebook — Enterprise Data Management

**What you'll learn:**
- Data governance fundamentals (compliance, discovery, quality, access)
- Unity Catalog architecture (conceptual — requires Enterprise)
- Access control patterns (GRANT/REVOKE, row/column security)
- Data classification & PII handling
- Lineage tracking, documentation, retention policies
- Audit logging

**Note:** Unity Catalog requires Databricks Enterprise. This notebook teaches
governance PATTERNS that work anywhere, with UC syntax shown conceptually.

**Prerequisites:** Notebooks 01-25 (techmart_dw)

---

---
# 📗 Section 1: Why Data Governance Matters

| Concern | Without Governance | With Governance |
|---|---|---|
| Compliance | GDPR fine (€20M+) | Documented, auditable |
| Discovery | "Where's the customer data?" | Self-service catalog |
| Quality | Silent data corruption | SLA-monitored, alerting |
| Access | Everyone sees everything | Least-privilege access |
| Lineage | "What broke?" (no idea) | Impact analysis in seconds |
| Cost | Uncontrolled growth | Retention policies, cleanup |

## The Cost of Poor Data Governance

### Real-World Failures

| Company | Incident | Cost |
|---------|---------|------|
| **Equifax (2017)** | Unpatched system, 147M records breached | $700M settlement |
| **Facebook (2018)** | Cambridge Analytica -- 87M user profiles misused | $5B FTC fine |
| **British Airways (2018)** | 500K customer records stolen | £20M GDPR fine |
| **Marriott (2018)** | 500M guest records exposed | $124M fine |
| **Capital One (2019)** | 100M customer records breached | $80M fine |

### What Good Governance Prevents

```
WITHOUT GOVERNANCE:                    WITH GOVERNANCE:
┌─────────────────────────┐           ┌─────────────────────────┐
│ "Who owns this table?"  │           │ Every table has an owner │
│ "Is this data fresh?"   │           │ SLAs tracked and alerted │
│ "Can I use this PII?"   │           │ PII tagged, access logged│
│ "Where did this come    │           │ Full lineage tracked     │
│  from?"                 │           │ automatically            │
│ "Is this data correct?" │           │ Quality checks enforced  │
│ "Who accessed this?"    │           │ Full audit trail         │
└─────────────────────────┘           └─────────────────────────┘
```

### The Four Pillars of Data Governance

| Pillar | What It Covers | Tools |
|--------|---------------|-------|
| **Discoverability** | Can people find the data they need? | Unity Catalog, DataHub, Amundsen |
| **Accessibility** | Can the right people access it? | RBAC, ABAC, row/column security |
| **Trustworthiness** | Is the data accurate and fresh? | Quality checks, SLAs, lineage |
| **Accountability** | Who is responsible for what? | Ownership, stewardship, audit logs |

---
# 📗 Section 2: Unity Catalog Architecture (Conceptual)

## Three-Level Namespace

```
METASTORE (one per region)
└── CATALOG (e.g., 'production', 'development', 'sandbox')
    └── SCHEMA (e.g., 'techmart_dw', 'healthcare_dw')
        └── TABLE / VIEW / FUNCTION
            └── COLUMN

Full reference: production.techmart_dw.orders
```

## Catalog Design for TechMart

```
production/                    development/                sandbox/
├── techmart_dw/              ├── techmart_dw/            ├── analyst_workspace/
│   ├── orders               │   ├── orders (sample)    │   └── (any table)
│   ├── customers            │   └── ...                └── ...
│   └── ...                  └── ...
├── healthcare_dw/
├── fintech_dw/
└── gold/
    ├── daily_sales
    └── customer_360
```

⚠️ Unity Catalog requires Databricks Enterprise (not Community Edition).

## Unity Catalog -- Detailed Architecture

### The Three-Level Namespace

```
METASTORE (one per Databricks account/region)
│
├── CATALOG: prod
│   ├── SCHEMA: sales
│   │   ├── TABLE: orders
│   │   ├── TABLE: customers
│   │   └── VIEW: active_customers
│   ├── SCHEMA: finance
│   │   ├── TABLE: transactions
│   │   └── TABLE: reconciliation
│   └── SCHEMA: ml_features
│       └── TABLE: customer_features
│
├── CATALOG: staging
│   └── (same structure, different data)
│
└── CATALOG: dev
    └── (same structure, sandbox data)

Access pattern: SELECT * FROM prod.sales.orders
                             ^^^^  ^^^^^  ^^^^^^
                            catalog schema  table
```

### Securable Objects Hierarchy

```
METASTORE
  └── CATALOG (USAGE required to see)
        └── SCHEMA (USAGE required to see)
              ├── TABLE (SELECT, MODIFY, ALL PRIVILEGES)
              ├── VIEW (SELECT)
              ├── FUNCTION (EXECUTE)
              └── VOLUME (READ_VOLUME, WRITE_VOLUME)
```

### Permission Inheritance

```
GRANT USAGE ON CATALOG prod TO `analyst-group`;
GRANT USAGE ON SCHEMA prod.sales TO `analyst-group`;
GRANT SELECT ON TABLE prod.sales.orders TO `analyst-group`;

-- analyst-group can now: SELECT * FROM prod.sales.orders
-- analyst-group CANNOT: INSERT, UPDATE, DELETE, DROP
-- analyst-group CANNOT: access prod.finance (no USAGE on that schema)
```

In [0]:
# Unity Catalog permission model simulation
class UnityCatalogPermissions:
    """Simulates Unity Catalog RBAC permission checks."""
    
    def __init__(self):
        self.grants = {}  # (principal, object_path) → set of privileges
        self.objects = {}  # path → type
    
    def grant(self, privilege, object_path, principal):
        key = (principal, object_path)
        if key not in self.grants:
            self.grants[key] = set()
        self.grants[key].add(privilege.upper())
    
    def revoke(self, privilege, object_path, principal):
        key = (principal, object_path)
        if key in self.grants:
            self.grants[key].discard(privilege.upper())
    
    def check(self, principal, object_path, privilege):
        """Check if principal has privilege on object (and all parents)."""
        # Check direct grant
        key = (principal, object_path)
        privs = self.grants.get(key, set())
        if privilege.upper() in privs or "ALL PRIVILEGES" in privs:
            return True
        
        # Check group membership (simplified)
        for (p, obj), ps in self.grants.items():
            if obj == object_path and p.endswith("-group"):
                if privilege.upper() in ps or "ALL PRIVILEGES" in ps:
                    return True
        return False
    
    def show_grants(self, object_path):
        """Show all grants on an object."""
        result = []
        for (principal, obj), privs in self.grants.items():
            if obj == object_path:
                result.append({"principal": principal, "privileges": list(privs)})
        return result


# Build a realistic permission model
uc = UnityCatalogPermissions()

# Setup: Grant permissions to groups
# Analysts: read-only on sales data
uc.grant("USAGE", "prod", "analyst-group")
uc.grant("USAGE", "prod.sales", "analyst-group")
uc.grant("SELECT", "prod.sales.orders", "analyst-group")
uc.grant("SELECT", "prod.sales.customers", "analyst-group")

# Engineers: full access to all schemas
uc.grant("USAGE", "prod", "engineer-group")
uc.grant("ALL PRIVILEGES", "prod.sales", "engineer-group")
uc.grant("ALL PRIVILEGES", "prod.finance", "engineer-group")

# Finance team: read finance, no sales
uc.grant("USAGE", "prod", "finance-group")
uc.grant("USAGE", "prod.finance", "finance-group")
uc.grant("SELECT", "prod.finance.transactions", "finance-group")

print("🔐 UNITY CATALOG PERMISSION CHECKS")
print("=" * 60)

checks = [
    ("analyst-group", "prod.sales.orders", "SELECT", "Should be allowed"),
    ("analyst-group", "prod.sales.orders", "INSERT", "Should be denied"),
    ("analyst-group", "prod.finance.transactions", "SELECT", "Should be denied"),
    ("engineer-group", "prod.sales.orders", "INSERT", "Should be allowed"),
    ("finance-group", "prod.finance.transactions", "SELECT", "Should be allowed"),
    ("finance-group", "prod.sales.orders", "SELECT", "Should be denied"),
    ("intern", "prod.sales.orders", "SELECT", "Should be denied"),
]

print(f"{'Principal':<20} {'Object':<35} {'Privilege':<10} {'Result':<10} {'Expected'}")
print("-" * 90)
for principal, obj, priv, expected in checks:
    allowed = uc.check(principal, obj, priv)
    result = "✅ ALLOW" if allowed else "❌ DENY"
    match = "✅" if (allowed and "allowed" in expected) or (not allowed and "denied" in expected) else "❌"
    print(f"{principal:<20} {obj:<35} {priv:<10} {result:<10} {match} {expected}")


---
# 📗 Section 3: Access Control Patterns

## SQL Syntax (Conceptual — requires Unity Catalog)

```sql
-- Table-level access
GRANT SELECT ON TABLE production.techmart_dw.orders TO `analysts`;
GRANT ALL PRIVILEGES ON SCHEMA production.techmart_dw TO `data_engineers`;
REVOKE SELECT ON TABLE production.techmart_dw.customers FROM `interns`;

-- Column masking (hide PII)
CREATE FUNCTION mask_email(email STRING)
RETURNS STRING
RETURN CONCAT(LEFT(email, 2), '***@', SPLIT(email, '@')[1]);

ALTER TABLE customers ALTER COLUMN email SET MASK mask_email;

-- Row-level security (filter by team)
CREATE FUNCTION region_filter(region STRING)
RETURNS BOOLEAN
RETURN (region = current_user_region() OR is_admin());

ALTER TABLE orders SET ROW FILTER region_filter ON (region);
```

## Access Control Matrix (Design Pattern)

| Role | Bronze | Silver | Gold | PII Columns |
|---|---|---|---|---|
| Data Engineer | Read/Write | Read/Write | Read/Write | Full access |
| Data Analyst | No access | Read only | Read only | Masked |
| BI Developer | No access | No access | Read only | Masked |
| Executive | No access | No access | Read only | No access |

## Row-Level and Column-Level Security

### Row-Level Security (RLS)

Restrict which ROWS a user can see based on their attributes:

```sql
-- Create a row filter function
CREATE FUNCTION prod.security.filter_by_region(region STRING)
RETURNS BOOLEAN
RETURN IF(IS_MEMBER('admin-group'), TRUE, region = CURRENT_USER_REGION());

-- Apply to table
ALTER TABLE prod.sales.orders
SET ROW FILTER prod.security.filter_by_region ON (region);

-- Now:
-- Admin sees ALL rows
-- User in 'US-East' sees only US-East rows
-- User in 'EU-West' sees only EU-West rows
```

### Column-Level Security (Data Masking)

Mask sensitive columns based on user role:

```sql
-- Create a masking function
CREATE FUNCTION prod.security.mask_email(email STRING)
RETURNS STRING
RETURN IF(IS_MEMBER('pii-access-group'), email, CONCAT(LEFT(email, 2), '***@***.com'));

-- Apply to column
ALTER TABLE prod.sales.customers
ALTER COLUMN email SET MASK prod.security.mask_email;

-- Now:
-- PII team sees: alice@example.com
-- Analyst sees:  al***@***.com
```

### Common Masking Patterns

| Data Type | Masking Pattern | Example |
|-----------|----------------|---------|
| Email | Show first 2 chars | `al***@***.com` |
| Phone | Show last 4 digits | `***-***-1234` |
| SSN | Show last 4 digits | `***-**-5678` |
| Credit card | Show last 4 digits | `****-****-****-1234` |
| Name | Show first initial | `A. Smith` |
| Address | Show city only | `New York, NY` |
| Salary | Round to nearest $10K | `$80,000` |

In [0]:
# Data masking simulation
import re

class DataMasker:
    """Simulates Unity Catalog column-level security masking."""
    
    def __init__(self, user_groups):
        self.user_groups = set(user_groups)
    
    def mask_email(self, email):
        if "pii-access-group" in self.user_groups:
            return email  # Full access
        if email and "@" in email:
            local, domain = email.split("@", 1)
            return f"{local[:2]}***@***.com"
        return "***@***.com"
    
    def mask_phone(self, phone):
        if "pii-access-group" in self.user_groups:
            return phone
        if phone:
            digits = re.sub(r"\D", "", phone)
            return f"***-***-{digits[-4:]}" if len(digits) >= 4 else "***-***-****"
        return "***-***-****"
    
    def mask_ssn(self, ssn):
        if "pii-access-group" in self.user_groups:
            return ssn
        if ssn:
            clean = re.sub(r"\D", "", ssn)
            return f"***-**-{clean[-4:]}" if len(clean) >= 4 else "***-**-****"
        return "***-**-****"
    
    def mask_credit_card(self, cc):
        if "pii-access-group" in self.user_groups:
            return cc
        if cc:
            clean = re.sub(r"\D", "", cc)
            return f"****-****-****-{clean[-4:]}" if len(clean) >= 4 else "****-****-****-****"
        return "****-****-****-****"
    
    def apply_to_record(self, record):
        """Apply all masking rules to a record."""
        masked = dict(record)
        if "email" in masked:
            masked["email"] = self.mask_email(masked["email"])
        if "phone" in masked:
            masked["phone"] = self.mask_phone(masked["phone"])
        if "ssn" in masked:
            masked["ssn"] = self.mask_ssn(masked["ssn"])
        if "credit_card" in masked:
            masked["credit_card"] = self.mask_credit_card(masked["credit_card"])
        return masked


# Test record
customer = {
    "customer_id": 42,
    "name": "Alice Smith",
    "email": "alice.smith@example.com",
    "phone": "555-867-5309",
    "ssn": "123-45-6789",
    "credit_card": "4111-1111-1111-1234",
    "city": "New York",
}

print("🔒 DATA MASKING SIMULATION")
print("=" * 60)

# Different user roles see different data
roles = [
    ("PII Access Team", ["pii-access-group", "analyst-group"]),
    ("Regular Analyst", ["analyst-group"]),
    ("External Auditor", []),
]

for role_name, groups in roles:
    masker = DataMasker(groups)
    masked = masker.apply_to_record(customer)
    print(f"\n👤 {role_name} (groups: {groups or ['none']}):")
    for key, value in masked.items():
        original = customer[key]
        changed = "🔒" if str(value) != str(original) else "  "
        print(f"   {changed} {key}: {value}")


---
# 📗 Section 4: Data Classification & Tagging

In [0]:
# Auto-classify columns by name pattern
import re

def classify_columns(table_name):
    """Classify columns by sensitivity based on naming patterns."""
    df = spark.table(table_name)
    
    pii_patterns = {
        "email": r"(?i)(email|e_mail)",
        "phone": r"(?i)(phone|mobile|cell|tel)",
        "name": r"(?i)(first_name|last_name|full_name|customer_name)",
        "address": r"(?i)(address|street|city|state|zip|postal)",
        "ssn": r"(?i)(ssn|social_security)",
        "dob": r"(?i)(dob|birth_date|date_of_birth)",
    }
    
    classifications = []
    for col_name in df.columns:
        sensitivity = "public"
        pii_type = None
        
        for pii_name, pattern in pii_patterns.items():
            if re.search(pattern, col_name):
                sensitivity = "confidential" if pii_name in ("ssn", "dob") else "internal"
                pii_type = pii_name
                break
        
        # ID columns are internal
        if col_name.endswith("_id") and not pii_type:
            sensitivity = "internal"
        
        # Audit columns are internal
        if col_name.startswith("_"):
            sensitivity = "internal"
        
        classifications.append({
            "column": col_name,
            "sensitivity": sensitivity,
            "pii_type": pii_type or "none",
            "dtype": str(df.schema[col_name].dataType)
        })
    
    return classifications

# Classify customers table
print("Column Classification: techmart_dw.customers")
print(f"{'Column':<25} {'Sensitivity':<15} {'PII Type':<12} {'Type'}")
print("-" * 70)
for c in classify_columns("techmart_dw.customers"):
    icon = {"confidential": "🔴", "internal": "🟡", "public": "🟢"}[c["sensitivity"]]
    print(f"{icon} {c['column']:<23} {c['sensitivity']:<15} {c['pii_type']:<12} {c['dtype'][:15]}")

---
# 📗 Section 5: Data Lineage

In [0]:
# Build a simple lineage tracker
import pandas as pd

class LineageTracker:
    """Track data lineage (source → target relationships)."""
    
    def __init__(self):
        self.lineage = []
    
    def add(self, source_table, target_table, transform_type, columns_used=None):
        self.lineage.append({
            "source": source_table,
            "target": target_table,
            "transform": transform_type,
            "columns": columns_used or "all"
        })
    
    def impact_analysis(self, table_name):
        """What downstream tables are affected if this table changes?"""
        affected = set()
        queue = [table_name]
        while queue:
            current = queue.pop(0)
            for entry in self.lineage:
                if entry["source"] == current and entry["target"] not in affected:
                    affected.add(entry["target"])
                    queue.append(entry["target"])
        return affected
    
    def show(self):
        print("Data Lineage:")
        for entry in self.lineage:
            print(f"  {entry['source']} ──[{entry['transform']}]──▶ {entry['target']}")

# Define lineage for our pipeline
lineage = LineageTracker()
lineage.add("techmart_dw.orders", "bronze_orders", "ingest")
lineage.add("techmart_dw.customers", "bronze_customers", "ingest")
lineage.add("bronze_orders", "silver_orders", "clean+dedup")
lineage.add("bronze_customers", "silver_customers", "clean+dedup")
lineage.add("silver_orders", "gold_daily_sales", "aggregate")
lineage.add("silver_orders", "gold_customer_360", "aggregate")
lineage.add("silver_customers", "gold_customer_360", "join")

lineage.show()
print(f"\nImpact of changing 'techmart_dw.orders': {lineage.impact_analysis('techmart_dw.orders')}")

## Data Lineage -- Tracking Data Flow

### What is Data Lineage?

Data lineage tracks WHERE data came from and WHERE it goes:

```
Source Systems          Bronze              Silver              Gold
┌──────────┐      ┌──────────┐      ┌──────────┐      ┌──────────┐
│ MySQL    │─────▶│raw_orders│─────▶│clean_    │─────▶│daily_    │
│ orders   │      │          │      │orders    │      │revenue   │
└──────────┘      └──────────┘      └──────────┘      └──────────┘
                                          │
                                          ▼
                                    ┌──────────┐
                                    │customer_ │─────▶ ML Model
                                    │segments  │
                                    └──────────┘
```

### Column-Level Lineage

Unity Catalog tracks lineage at the COLUMN level:

```
orders.total_amount
    ↓ (SUM)
daily_revenue.total_revenue
    ↓ (used in)
finance_dashboard.revenue_chart
```

This means you can answer: "If I change the definition of total_amount, what breaks?"

### Lineage Use Cases

| Use Case | How Lineage Helps |
|----------|------------------|
| **Impact analysis** | "What breaks if I change this column?" |
| **Root cause analysis** | "Where did this wrong value come from?" |
| **Compliance** | "Show me all tables that contain PII" |
| **Data discovery** | "What tables use this source system?" |
| **Debugging** | "Why is this Gold table wrong?" |

### Querying Lineage in Unity Catalog

```sql
-- Find all tables that read from a specific table
SELECT * FROM system.access.table_lineage
WHERE source_table_full_name = 'prod.sales.orders';

-- Find all tables that write to a specific table
SELECT * FROM system.access.table_lineage
WHERE target_table_full_name = 'prod.gold.daily_revenue';

-- Audit: who accessed what data
SELECT * FROM system.access.audit
WHERE action_name = 'SELECT'
  AND request_params.table_full_name = 'prod.sales.customers'
ORDER BY event_time DESC
LIMIT 100;
```

In [0]:
# Data lineage simulation
from collections import defaultdict

class DataLineageTracker:
    """Simulates Unity Catalog data lineage tracking."""
    
    def __init__(self):
        self.lineage = defaultdict(list)  # source → [targets]
        self.reverse_lineage = defaultdict(list)  # target → [sources]
        self.column_lineage = {}  # (target_table, target_col) → [(source_table, source_col)]
    
    def register_transformation(self, source_tables, target_table, column_mappings=None):
        """Register a transformation that creates target from sources."""
        for source in source_tables:
            self.lineage[source].append(target_table)
            self.reverse_lineage[target_table].append(source)
        
        if column_mappings:
            for target_col, source_info in column_mappings.items():
                self.column_lineage[(target_table, target_col)] = source_info
    
    def get_downstream(self, table, depth=0, max_depth=5):
        """Get all downstream tables (what depends on this table)."""
        if depth >= max_depth:
            return []
        result = []
        for downstream in self.lineage.get(table, []):
            result.append((downstream, depth + 1))
            result.extend(self.get_downstream(downstream, depth + 1, max_depth))
        return result
    
    def get_upstream(self, table, depth=0, max_depth=5):
        """Get all upstream tables (what this table depends on)."""
        if depth >= max_depth:
            return []
        result = []
        for upstream in self.reverse_lineage.get(table, []):
            result.append((upstream, depth + 1))
            result.extend(self.get_upstream(upstream, depth + 1, max_depth))
        return result
    
    def impact_analysis(self, table):
        """What breaks if I change this table?"""
        downstream = self.get_downstream(table)
        return [t for t, _ in downstream]


# Build a realistic lineage graph
tracker = DataLineageTracker()

# Source systems → Bronze
tracker.register_transformation(["mysql.orders"], "bronze.raw_orders")
tracker.register_transformation(["mysql.customers"], "bronze.raw_customers")
tracker.register_transformation(["kafka.payments"], "bronze.raw_payments")

# Bronze → Silver
tracker.register_transformation(
    ["bronze.raw_orders", "bronze.raw_customers"],
    "silver.clean_orders",
    column_mappings={
        "total_revenue": [("bronze.raw_orders", "total_amount")],
        "customer_name": [("bronze.raw_customers", "first_name"), ("bronze.raw_customers", "last_name")],
    }
)
tracker.register_transformation(["bronze.raw_payments"], "silver.clean_payments")

# Silver → Gold
tracker.register_transformation(
    ["silver.clean_orders"],
    "gold.daily_revenue",
    column_mappings={"revenue": [("silver.clean_orders", "total_revenue")]}
)
tracker.register_transformation(
    ["silver.clean_orders", "silver.clean_payments"],
    "gold.payment_reconciliation"
)
tracker.register_transformation(["silver.clean_orders"], "gold.customer_segments")

# Gold → Dashboards/ML
tracker.register_transformation(["gold.daily_revenue"], "dashboard.finance_report")
tracker.register_transformation(["gold.customer_segments"], "ml_model.churn_predictor")

print("📊 DATA LINEAGE ANALYSIS")
print("=" * 60)

# Impact analysis: what breaks if mysql.orders changes?
print("\n🔍 Impact Analysis: What breaks if 'mysql.orders' changes?")
impacted = tracker.impact_analysis("mysql.orders")
for table in impacted:
    print(f"   ⚠️ {table}")

# Upstream analysis: where does gold.daily_revenue come from?
print("\n🔍 Upstream: Where does 'gold.daily_revenue' come from?")
upstream = tracker.get_upstream("gold.daily_revenue")
for table, depth in upstream:
    indent = "  " * depth
    print(f"   {indent}← {table}")

# Column lineage
print("\n🔍 Column Lineage: Where does 'gold.daily_revenue.revenue' come from?")
col_lineage = tracker.column_lineage.get(("gold.daily_revenue", "revenue"), [])
for source_table, source_col in col_lineage:
    print(f"   {source_table}.{source_col} → gold.daily_revenue.revenue")


---
# 📗 Section 6: Data Documentation

In [0]:
%sql
-- Add table and column comments (works on Community Edition!)
USE techmart_dw;

COMMENT ON TABLE orders IS 'E-commerce order transactions. Grain: one row per order. Source: techmart_oltp.';
COMMENT ON TABLE customers IS 'Customer master data with intentional quality issues for practice.';

-- View table comment
DESCRIBE TABLE EXTENDED orders;

In [0]:
# Auto-generate data dictionary
import pandas as pd

def generate_data_dictionary(database="techmart_dw"):
    """Generate a data dictionary for all tables in a database."""
    tables = spark.sql(f"SHOW TABLES IN {database}").collect()
    
    dictionary = []
    for table_row in tables[:10]:  # Limit for demo
        table_name = table_row.tableName
        try:
            df = spark.table(f"{database}.{table_name}")
            row_count = df.count()
            
            for field in df.schema.fields:
                dictionary.append({
                    "database": database,
                    "table": table_name,
                    "column": field.name,
                    "data_type": str(field.dataType),
                    "nullable": field.nullable,
                    "table_rows": row_count
                })
        except Exception:
            pass
    
    return pd.DataFrame(dictionary)

# Generate dictionary
dict_df = generate_data_dictionary("techmart_dw")
print(f"Data Dictionary: {len(dict_df)} column entries across {dict_df['table'].nunique()} tables")
print(dict_df.groupby("table").size().sort_values(ascending=False).head(10))

---
# 📗 Section 7: PII Handling & Masking

In [0]:
# Implement masking functions (runnable pattern)
from pyspark.sql.functions import *

customers = spark.table("techmart_dw.customers")

# Masking strategies
masked = (customers
    # Email: show first 2 chars + domain
    .withColumn("email_masked",
        when(col("email").isNotNull(),
            concat(substring("email", 1, 2), lit("***@"), regexp_extract("email", r"@(.+)$", 1)))
        .otherwise(lit(None)))
    # Phone: show last 4 digits
    .withColumn("phone_masked",
        when(col("phone").isNotNull(), concat(lit("***-***-"), substring("phone", -4, 4)))
        .otherwise(lit(None)))
    # Name: first initial + last name
    .withColumn("name_masked",
        concat(substring("first_name", 1, 1), lit(". "), col("last_name")))
)

print("PII Masking Demo:")
masked.select("customer_id", "name_masked", "email_masked", "phone_masked").show(5, truncate=False)

## Data Contracts -- Producer-Consumer Agreements

### What is a Data Contract?

A data contract is a formal agreement between a data producer (who creates data)
and data consumers (who use it). It defines:

- **Schema**: What columns exist, what types they are
- **Quality**: What quality guarantees are made
- **SLAs**: How fresh the data will be
- **Ownership**: Who is responsible
- **Breaking changes**: How changes will be communicated

### Data Contract Example (YAML)

```yaml
# data_contract.yml
contract:
  name: "orders_daily"
  version: "2.1.0"
  owner: "order-service-team@company.com"
  consumers:
    - "analytics-team"
    - "ml-team"
    - "finance-team"
  
  schema:
    - name: order_id
      type: integer
      nullable: false
      unique: true
      description: "Unique order identifier"
    
    - name: customer_id
      type: integer
      nullable: false
      description: "FK to customers table"
    
    - name: total_amount
      type: decimal(12,2)
      nullable: false
      constraints:
        min: 0
        max: 1000000
      description: "Order total in USD"
    
    - name: status
      type: string
      nullable: false
      allowed_values: ["pending", "completed", "shipped", "cancelled"]
      description: "Current order status"
    
    - name: order_date
      type: date
      nullable: false
      description: "Date order was placed"
  
  quality:
    completeness: ">= 99.9%"
    uniqueness: "order_id is unique"
    freshness: "Updated within 2 hours"
    accuracy: "Reconciled with source system daily"
  
  sla:
    availability: "99.5%"
    freshness: "2 hours"
    support_response: "4 hours for P1 issues"
  
  breaking_changes:
    notice_period: "14 days"
    approval_required: true
    notification_channels: ["#data-engineering", "email"]
  
  changelog:
    - version: "2.1.0"
      date: "2024-03-15"
      changes: "Added order_source column"
    - version: "2.0.0"
      date: "2024-01-01"
      changes: "Breaking: renamed amount to total_amount"
```

In [0]:
# Data contract validation
import json
from datetime import datetime

class DataContract:
    """Validates data against a data contract."""
    
    def __init__(self, contract_definition):
        self.contract = contract_definition
        self.violations = []
    
    def validate(self, data):
        """Validate a dataset against the contract."""
        self.violations = []
        
        schema = {col["name"]: col for col in self.contract["schema"]}
        
        for i, record in enumerate(data):
            for col_name, col_def in schema.items():
                value = record.get(col_name)
                
                # Null check
                if not col_def.get("nullable", True) and value is None:
                    self.violations.append({
                        "row": i, "column": col_name,
                        "rule": "not_null", "value": value
                    })
                    continue
                
                if value is None:
                    continue
                
                # Type check
                expected_type = col_def.get("type", "")
                if "integer" in expected_type and not isinstance(value, int):
                    self.violations.append({
                        "row": i, "column": col_name,
                        "rule": "type_check", "value": value,
                        "expected": "integer"
                    })
                
                # Allowed values check
                if "allowed_values" in col_def and value not in col_def["allowed_values"]:
                    self.violations.append({
                        "row": i, "column": col_name,
                        "rule": "allowed_values", "value": value,
                        "allowed": col_def["allowed_values"]
                    })
                
                # Range check
                constraints = col_def.get("constraints", {})
                if "min" in constraints and isinstance(value, (int, float)):
                    if value < constraints["min"]:
                        self.violations.append({
                            "row": i, "column": col_name,
                            "rule": "min_value", "value": value,
                            "min": constraints["min"]
                        })
        
        return len(self.violations) == 0
    
    def report(self):
        if not self.violations:
            print("   ✅ All contract checks PASSED")
        else:
            print(f"   ❌ {len(self.violations)} contract violations found:")
            for v in self.violations[:5]:
                print(f"      Row {v['row']}, {v['column']}: {v['rule']} -- value={v['value']}")


# Define a contract
orders_contract = {
    "name": "orders_daily",
    "schema": [
        {"name": "order_id", "type": "integer", "nullable": False},
        {"name": "customer_id", "type": "integer", "nullable": False},
        {"name": "total_amount", "type": "decimal", "nullable": False,
         "constraints": {"min": 0, "max": 1000000}},
        {"name": "status", "type": "string", "nullable": False,
         "allowed_values": ["pending", "completed", "shipped", "cancelled"]},
    ]
}

# Test data (with some violations)
test_data = [
    {"order_id": 1, "customer_id": 42, "total_amount": 99.99, "status": "completed"},
    {"order_id": None, "customer_id": 17, "total_amount": 50.00, "status": "pending"},  # NULL order_id
    {"order_id": 3, "customer_id": 55, "total_amount": -10.00, "status": "shipped"},    # Negative amount
    {"order_id": 4, "customer_id": 88, "total_amount": 200.00, "status": "INVALID"},    # Bad status
    {"order_id": 5, "customer_id": 33, "total_amount": 75.00, "status": "cancelled"},
]

print("📋 DATA CONTRACT VALIDATION")
print("=" * 60)
print(f"Contract: {orders_contract['name']}")
print(f"Records to validate: {len(test_data)}")

contract = DataContract(orders_contract)
is_valid = contract.validate(test_data)
contract.report()

print(f"\n   Contract status: {'✅ VALID' if is_valid else '❌ INVALID'}")
print("\n💡 In production:")
print("   • Contracts are stored in a schema registry")
print("   • Validated automatically in CI/CD pipelines")
print("   • Breaking changes require consumer approval")
print("   • Violations trigger alerts to data owners")


---
# 🚀 Section 8: Mini Project — Data Catalog

In [0]:
# Self-service data catalog
import pandas as pd
from datetime import datetime

def build_catalog(database="techmart_dw", max_tables=15):
    """Build a self-service data catalog."""
    tables = spark.sql(f"SHOW TABLES IN {database}").collect()
    catalog = []
    
    for t in tables[:max_tables]:
        table_name = t.tableName
        try:
            df = spark.table(f"{database}.{table_name}")
            row_count = df.count()
            col_count = len(df.columns)
            
            # Classify PII
            pii_cols = [c for c in df.columns if any(p in c.lower() for p in ["email", "phone", "name", "address"])]
            
            catalog.append({
                "database": database,
                "table": table_name,
                "rows": row_count,
                "columns": col_count,
                "pii_columns": len(pii_cols),
                "pii_column_names": ", ".join(pii_cols) if pii_cols else "none",
                "has_pii": len(pii_cols) > 0,
                "cataloged_at": datetime.now().isoformat()
            })
        except Exception:
            pass
    
    catalog_df = pd.DataFrame(catalog)
    
    print(f"{'='*70}")
    print(f"DATA CATALOG: {database}")
    print(f"{'='*70}")
    print(f"Tables: {len(catalog_df)} | Total rows: {catalog_df['rows'].sum():,}")
    print(f"Tables with PII: {catalog_df['has_pii'].sum()}")
    print(f"\n{'Table':<30} {'Rows':<10} {'Cols':<6} {'PII':<5} {'PII Columns'}")
    print("-" * 70)
    for _, row in catalog_df.iterrows():
        pii_flag = "⚠️" if row["has_pii"] else "✅"
        print(f"{pii_flag} {row['table']:<28} {row['rows']:<10,} {row['columns']:<6} {row['pii_columns']:<5} {row['pii_column_names'][:25]}")
    print(f"{'='*70}")
    
    return catalog_df

catalog = build_catalog("techmart_dw")

---
# 🏆 Section 9: Interview Questions

## Q1: How do you handle PII in a data lake?

1. **Classify:** Identify PII columns (auto-scan by name pattern + manual review)
2. **Tag:** Mark columns with sensitivity level (confidential, internal, public)
3. **Mask:** Apply masking for non-privileged users (hash, redact, generalize)
4. **Access control:** Restrict raw PII to authorized roles only
5. **Audit:** Log all access to PII columns
6. **Retention:** Delete PII per GDPR right-to-be-forgotten requests

## Q2: Unity Catalog three-level namespace?

`catalog.schema.table` — like a file system:
- **Catalog:** Top-level container (production, development, sandbox)
- **Schema:** Logical grouping (techmart_dw, healthcare_dw)
- **Table:** The actual data object

Benefits: fine-grained access control at any level, cross-workspace sharing, centralized governance.

## Q3: Row-level security?

Define a function that returns TRUE/FALSE based on the current user's attributes.
Apply it as a row filter: `ALTER TABLE SET ROW FILTER func ON (column)`.
Each user only sees rows where the function returns TRUE.

## Q4: Data lineage approach?

1. **Automatic:** Unity Catalog tracks lineage from Spark queries automatically
2. **Manual:** Maintain a lineage metadata table (source → target mappings)
3. **Impact analysis:** Given a source change, trace all downstream dependencies
4. **Column-level:** Track which source columns feed which target columns

## Q5: GDPR compliance?

1. **Data inventory:** Know where all personal data lives
2. **Consent tracking:** Record what processing is consented
3. **Right to access:** Export all data for a given person
4. **Right to delete:** Delete/anonymize all data for a person
5. **Data minimization:** Only collect what's needed
6. **Audit trail:** Prove compliance to regulators

## Q6: Data classification strategy?

- **Auto-classify:** Scan column names for PII patterns (email, phone, ssn)
- **Manual review:** Data stewards validate and override auto-classification
- **Levels:** Public → Internal → Confidential → Restricted
- **Enforcement:** Access controls aligned to classification level
- **Regular audit:** Re-scan quarterly for new tables/columns

---
# 📗 Section 9: Governance in Practice

## Building a Data Catalog

A data catalog is the "Google for your data" -- helps people find and understand data.

### What a Good Catalog Entry Looks Like

```yaml
table: prod.sales.orders
description: "All customer orders since 2020-01-01. Updated hourly."
owner: "order-service-team@company.com"
steward: "alice@company.com"
tags: ["production", "pii-adjacent", "sla-critical"]
sla:
  freshness: "1 hour"
  availability: "99.5%"
columns:
  - name: order_id
    type: INT
    description: "Unique order identifier"
    nullable: false
  - name: customer_id
    type: INT
    description: "FK to customers table"
    nullable: false
  - name: total_amount
    type: DECIMAL(12,2)
    description: "Order total in USD, excluding tax"
    nullable: false
lineage:
  upstream: ["mysql.ecommerce.orders"]
  downstream: ["gold.daily_revenue", "ml.churn_features"]
```

## GDPR/CCPA Compliance Checklist

| Requirement | Implementation |
|-------------|---------------|
| **Right to be forgotten** | Delete customer data on request (soft delete + VACUUM) |
| **Data portability** | Export customer data in machine-readable format |
| **Consent tracking** | Track when/how consent was given |
| **Data minimization** | Only collect what you need |
| **Purpose limitation** | Only use data for stated purpose |
| **Breach notification** | Alert within 72 hours of breach |

## Data Retention Policies

```sql
-- Implement retention: delete records older than 7 years
DELETE FROM prod.finance.transactions
WHERE transaction_date < DATE_SUB(CURRENT_DATE(), 2555);  -- 7 years

-- Or use Delta Lake time travel + VACUUM
ALTER TABLE prod.finance.transactions
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 7 years');
```

In [0]:
# Data governance framework
print("Data Governance Framework")
print("=" * 60)

governance_framework = {
    "Data Catalog": {
        "purpose": "Discover and understand data assets",
        "tools": ["Unity Catalog", "DataHub", "Amundsen", "Atlan"],
        "key_features": ["Search", "Lineage", "Ownership", "Documentation"],
    },
    "Access Control": {
        "purpose": "Ensure right people access right data",
        "tools": ["Unity Catalog RBAC", "Row/Column security"],
        "key_features": ["RBAC", "ABAC", "Row filters", "Column masking"],
    },
    "Data Quality": {
        "purpose": "Ensure data is accurate and reliable",
        "tools": ["Great Expectations", "dbt tests", "DLT expectations"],
        "key_features": ["Completeness", "Accuracy", "Freshness", "Uniqueness"],
    },
    "Data Lineage": {
        "purpose": "Track data flow and impact",
        "tools": ["Unity Catalog", "OpenLineage", "Marquez"],
        "key_features": ["Table lineage", "Column lineage", "Job lineage"],
    },
    "Compliance": {
        "purpose": "Meet regulatory requirements",
        "tools": ["Unity Catalog audit logs", "Custom compliance checks"],
        "key_features": ["GDPR", "CCPA", "HIPAA", "SOX", "PCI-DSS"],
    },
}

for pillar, details in governance_framework.items():
    print(f"\n{pillar}:")
    print(f"  Purpose: {details['purpose']}")
    print(f"  Tools: {', '.join(details['tools'])}")
    print(f"  Features: {', '.join(details['key_features'])}")


---
# ✅ Notebook Complete!

**What was covered:**
- Governance fundamentals: compliance, discovery, quality, access, lineage
- Unity Catalog architecture (conceptual): three-level namespace, managed tables
- Access control: GRANT/REVOKE, column masking, row-level security
- Data classification: auto-classify by column name patterns
- Lineage tracking: LineageTracker with impact analysis
- Documentation: table comments, auto-generated data dictionary
- PII masking: email, phone, name masking functions
- Mini project: self-service data catalog with PII detection
- 6 interview questions

**Next:** Notebook 27 — Data Quality Framework

---